In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🗓️ Meeting Transcript Summarizer

## What We're Building

A tool that takes a raw meeting transcript and produces a structured summary with:
- **Executive Summary** — What the meeting was about (2-3 sentences)
- **Key Decisions** — What was decided
- **Action Items** — Who does what by when
- **Blockers** — Open issues or risks
- **Open Questions** — Things that still need answers

## Key LangChain Concept: Output Parsers

LangChain **output parsers** transform raw LLM text into structured Python objects.
We'll use `PydanticOutputParser` to get perfectly typed, validated output.

```
LLM Output (text) → Parser → Python Object (typed, validated)
```

## Stack
- **LangChain** — chains with output parsers
- **Pydantic** — output schema definition
- **Ollama (qwen3.5:9b)** — local LLM

## Step 1 — Install & Import Dependencies

In [ ]:
# Install if needed (uncomment)
# !pip install langchain langchain-ollama -q

from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
import json

print("All imports successful!")

## Step 2 — Define the Meeting Summary Schema

We define a Pydantic model that captures everything we want from the summary.
This acts as a **contract** — the LLM must fill all these fields.

In [ ]:
class ActionItem(BaseModel):
    """A single action item from the meeting."""
    owner: str = Field(description="Person responsible")
    task: str = Field(description="What needs to be done")
    deadline: str = Field(description="When it's due (or 'TBD' if not specified)")
    priority: str = Field(description="high, medium, or low")


class Decision(BaseModel):
    """A decision made during the meeting."""
    decision: str = Field(description="What was decided")
    rationale: str = Field(description="Why this decision was made")


class MeetingSummary(BaseModel):
    """Structured summary of a meeting."""
    title: str = Field(description="Meeting title or topic")
    date: str = Field(description="Meeting date if mentioned, else 'Not specified'")
    attendees: list[str] = Field(description="List of people who participated")
    executive_summary: str = Field(
        description="2-3 sentence overview of the entire meeting"
    )
    decisions: list[Decision] = Field(
        description="Key decisions made during the meeting"
    )
    action_items: list[ActionItem] = Field(
        description="Tasks assigned to specific people"
    )
    blockers: list[str] = Field(
        description="Issues or risks that could block progress"
    )
    open_questions: list[str] = Field(
        description="Questions that remain unanswered"
    )
    next_meeting: str = Field(
        description="When the next meeting is, or 'Not discussed'"
    )


print("Schema defined!")
print(f"\nMeetingSummary has {len(MeetingSummary.model_fields)} fields:")
for name, field in MeetingSummary.model_fields.items():
    print(f"  - {name}: {field.annotation.__name__ if hasattr(field.annotation, '__name__') else field.annotation}")

## Step 3 — Sample Meeting Transcript

Here's a realistic meeting transcript for testing.

In [ ]:
sample_transcript = """
Sprint Planning Meeting - March 15, 2025

Participants: Sarah (PM), Mike (Tech Lead), Lisa (Frontend), James (Backend), 
              Priya (QA), Tom (Designer)

Sarah: Good morning everyone. Let's review where we stand on the Q1 release. 
Mike, can you give us the engineering update?

Mike: Sure. The API migration is about 70% done. We hit a snag with the 
authentication service — the OAuth2 integration with the new provider is 
taking longer than expected. James has been leading that effort.

James: Yeah, the main issue is that their sandbox environment has been 
unreliable. I've been in contact with their support team. They promised a 
fix by Wednesday. If that doesn't come through, we might need to consider 
the fallback provider we evaluated last month.

Sarah: That's concerning. What's the impact on the timeline if we switch?

Mike: Roughly 5 business days of additional work. We'd push the release 
from March 28 to April 4.

Sarah: OK, let's make a decision on this by Friday. James, keep pushing 
with the current provider until Wednesday. If their fix doesn't land, 
we'll go with the fallback. Mike, can you prepare the fallback 
implementation plan just in case?

Mike: Will do. I'll have it ready by Thursday.

Sarah: Lisa, how's the frontend looking?

Lisa: The new dashboard is feature-complete. I'm waiting on Tom's final 
designs for the settings page. Tom, any ETA on that?

Tom: I'll have the mockups done by end of day tomorrow. Sorry for the 
delay — I was pulled into the marketing site redesign last week.

Lisa: No worries. Once I get the designs, I'll need about 3 days to 
implement and test.

Sarah: Priya, how's the test plan?

Priya: I've written test cases for 80% of the new features. The main 
gap is the OAuth flow — I need the integration working to write proper 
E2E tests. I also found 3 critical bugs from last sprint that still 
aren't fixed.

Mike: Which bugs?

Priya: The memory leak in the notification service, the race condition 
in the payment queue, and the timezone display bug for European users. 
The first two are P0s.

Sarah: James and Mike, can you prioritize those P0 bugs this sprint? 
We can't ship with a memory leak.

Mike: Agreed. James, take the payment queue fix. I'll handle the 
notification service memory leak.

James: Got it.

Sarah: Great. Let's plan to reconvene on Wednesday to check the OAuth 
status. If anyone hits a blocker before then, ping me on Slack. Anything 
else?

Tom: Quick question — are we still planning the user onboarding redesign 
for Q2? I want to start research next week.

Sarah: Yes, that's still on the roadmap. Let's discuss it in next week's 
planning session. Good point bringing it up.

Sarah: Alright, that's a wrap. Thanks everyone!
"""

print(f"Transcript length: {len(sample_transcript)} characters")
print(f"Word count: {len(sample_transcript.split())} words")

## Step 4 — Build the Summarization Chain

In [ ]:
# Initialize LLM and parser
llm = ChatOllama(model="qwen3.5:9b", temperature=0.2)
parser = JsonOutputParser(pydantic_object=MeetingSummary)

# Build the prompt
summary_prompt = ChatPromptTemplate.from_template(
    """You are a professional meeting secretary. Analyze the following meeting
transcript and produce a structured summary.

Be precise:
- Extract exact names and dates mentioned
- Only include decisions that were actually agreed upon
- For action items, include the specific person, task, and deadline
- List blockers that could delay progress
- Capture questions that were raised but not fully answered

{format_instructions}

Meeting Transcript:
---
{transcript}
---

Structured summary as JSON:"""
)

# Build the chain
summary_chain = summary_prompt | llm | parser

print("Summarization chain ready!")

## Step 5 — Generate the Summary

In [ ]:
print("🔄 Analyzing transcript...\n")

summary = summary_chain.invoke({
    "transcript": sample_transcript,
    "format_instructions": parser.get_format_instructions(),
})

print("✅ Summary generated!\n")
print(json.dumps(summary, indent=2))

## Step 6 — Display as a Readable Report

In [ ]:
def display_summary(summary: dict) -> None:
    """Display the meeting summary in a human-readable format."""
    print("=" * 70)
    print(f"📋 {summary.get('title', 'Meeting Summary')}")
    print(f"📅 Date: {summary.get('date', 'N/A')}")
    print(f"👥 Attendees: {', '.join(summary.get('attendees', []))}")
    print("=" * 70)
    
    # Executive Summary
    print(f"\n📌 EXECUTIVE SUMMARY")
    print(f"   {summary.get('executive_summary', '')}")
    
    # Decisions
    decisions = summary.get('decisions', [])
    if decisions:
        print(f"\n✅ DECISIONS ({len(decisions)})")
        for d in decisions:
            if isinstance(d, dict):
                print(f"   • {d.get('decision', '')}")
                print(f"     Rationale: {d.get('rationale', '')}")
    
    # Action Items
    actions = summary.get('action_items', [])
    if actions:
        print(f"\n📝 ACTION ITEMS ({len(actions)})")
        for a in actions:
            if isinstance(a, dict):
                priority = a.get('priority', 'medium').upper()
                icon = '🔴' if priority == 'HIGH' else '🟡' if priority == 'MEDIUM' else '🟢'
                print(f"   {icon} [{priority}] {a.get('owner', '?')}: {a.get('task', '')}")
                print(f"      Due: {a.get('deadline', 'TBD')}")
    
    # Blockers
    blockers = summary.get('blockers', [])
    if blockers:
        print(f"\n🚧 BLOCKERS ({len(blockers)})")
        for b in blockers:
            print(f"   ⚠️ {b}")
    
    # Open Questions
    questions = summary.get('open_questions', [])
    if questions:
        print(f"\n❓ OPEN QUESTIONS ({len(questions)})")
        for q in questions:
            print(f"   • {q}")
    
    # Next Meeting
    print(f"\n📅 NEXT MEETING: {summary.get('next_meeting', 'Not discussed')}")
    print("=" * 70)


display_summary(summary)

## Step 7 — Export to Markdown

In [ ]:
from pathlib import Path


def export_to_markdown(summary: dict, output_path: str = "meeting_summary.md") -> None:
    """Export summary to a Markdown file."""
    lines = [
        f"# {summary.get('title', 'Meeting Summary')}",
        f"",
        f"**Date:** {summary.get('date', 'N/A')}",
        f"**Attendees:** {', '.join(summary.get('attendees', []))}",
        f"",
        f"## Executive Summary",
        f"{summary.get('executive_summary', '')}",
        f"",
    ]
    
    decisions = summary.get('decisions', [])
    if decisions:
        lines.append("## Decisions")
        for d in decisions:
            if isinstance(d, dict):
                lines.append(f"- **{d.get('decision', '')}**: {d.get('rationale', '')}")
        lines.append("")
    
    actions = summary.get('action_items', [])
    if actions:
        lines.append("## Action Items")
        lines.append("| Owner | Task | Deadline | Priority |")
        lines.append("|-------|------|----------|----------|")
        for a in actions:
            if isinstance(a, dict):
                lines.append(
                    f"| {a.get('owner', '')} | {a.get('task', '')} "
                    f"| {a.get('deadline', 'TBD')} | {a.get('priority', 'medium')} |"
                )
        lines.append("")
    
    blockers = summary.get('blockers', [])
    if blockers:
        lines.append("## Blockers")
        for b in blockers:
            lines.append(f"- ⚠️ {b}")
        lines.append("")
    
    questions = summary.get('open_questions', [])
    if questions:
        lines.append("## Open Questions")
        for q in questions:
            lines.append(f"- {q}")
        lines.append("")
    
    lines.append(f"**Next Meeting:** {summary.get('next_meeting', 'Not discussed')}")
    
    Path(output_path).write_text("\n".join(lines), encoding="utf-8")
    print(f"📄 Exported to: {output_path}")


export_to_markdown(summary)

## Step 8 — Try with Your Own Transcript

In [ ]:
def summarize_meeting(transcript: str) -> dict:
    """Complete pipeline: transcript → structured summary."""
    print("🔄 Processing transcript...")
    print(f"   Length: {len(transcript)} chars, {len(transcript.split())} words")
    
    result = summary_chain.invoke({
        "transcript": transcript,
        "format_instructions": parser.get_format_instructions(),
    })
    
    display_summary(result)
    return result


# Uncomment to try with your own transcript:
# my_transcript = """Paste your meeting transcript here..."""
# result = summarize_meeting(my_transcript)

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Pydantic Models** | Define exact output structure with nested types |
| **JsonOutputParser** | Parses LLM text into validated Python dicts |
| **Nested Models** | `ActionItem` inside `MeetingSummary` for deep structure |
| **format_instructions** | Auto-generated text that tells the LLM the expected format |
| **Post-processing** | Convert structured data to Markdown tables, reports |

## 🔧 Customization Ideas

- **Audio input**: Add Whisper for speech-to-text before summarization
- **Follow-up tracking**: Store action items in a database and track completion
- **Email integration**: Auto-send the summary to all attendees
- **Multi-meeting context**: Reference previous meetings' action items